[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/computer-vision/blob/main/implementation/notebooks/Computer_Vision_From_Scratch.ipynb)

# Computer Vision From Scratch — Convolution by Hand

**Companion notebook to** [`playbooks/ai/computer-vision/02_Concepts.md`](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/computer-vision/02_Concepts.md).

This notebook builds a CNN's forward pass **using only NumPy** — no PyTorch convolution, no autograd. Every patch is multiplied by hand. Every feature-map cell is computed explicitly. Every pooling window is reduced manually.

**Why this matters.** When you write `nn.Conv2d(...)` in PyTorch, the framework runs exactly what is in this notebook — just on bigger images with more filters. If the math here makes sense, the math at scale makes sense.

## What you will see
1. A tiny 5×5 image and a 3×3 vertical-edge filter
2. Manual single-position convolution (one slide computed by hand)
3. The full feature map built by sliding everywhere
4. **Verification** that PyTorch's `F.conv2d` produces the same numbers
5. ReLU activation, applied element-wise
6. Manual 2×2 max pooling
7. Multi-filter convolution — how depth works
8. Parameter counting verified against PyTorch's `model.parameters()`

By the end, `nn.Conv2d` will stop being magic.

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

np.set_printoptions(precision=2, suppress=True)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

## 1. The Tiny Image

A 5×5 grayscale image — think of it as a tiny digit-stroke pattern.

```
0 0 1 1 0
0 1 1 0 0
1 1 0 0 0
0 1 1 1 0
0 0 1 0 0
```

Real MNIST is 28×28. We use 5×5 so every step is computable by hand.

In [ ]:
image = np.array([
    [0, 0, 1, 1, 0],
    [0, 1, 1, 0, 0],
    [1, 1, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 0, 0],
], dtype=np.float32)

print(f"Image shape: {image.shape}")
print(image)

## 2. The Filter — A Vertical-Edge Detector

A 3×3 filter (also called a kernel). Positive on the left column, negative on the right column. This pattern fires when the input has bright pixels on the left and dark pixels on the right — a vertical edge.

```
 1  0 -1
 1  0 -1
 1  0 -1
```

The bias is +1 (added after the dot product, before the activation).

**Important.** This filter is hand-designed for the demo. In a real CNN, the filter values are *learned* during training — the network discovers which patterns matter from the data.

In [ ]:
filter_F = np.array([
    [ 1, 0, -1],
    [ 1, 0, -1],
    [ 1, 0, -1],
], dtype=np.float32)

bias = 1.0

print(f"Filter shape: {filter_F.shape}")
print(filter_F)
print(f"Bias: {bias}")

## 3. Manual Single Slide — Top-Left Position

Take the top-left 3×3 patch of the image. Multiply element-by-element with the filter. Sum. Add bias. Apply ReLU.

This is one cell of the output feature map.

In [ ]:
# Top-left 3x3 patch
patch = image[0:3, 0:3]
print("Patch:")
print(patch)
print()
print("Filter:")
print(filter_F)
print()

# Element-wise multiplication
elementwise = patch * filter_F
print("Element-wise multiplication:")
print(elementwise)
print()

# Sum
total = elementwise.sum()
print(f"Sum: {total}")

# Add bias
z = total + bias
print(f"+ bias = {z}")

# Apply ReLU
output = max(0, z)
print(f"ReLU({z}) = {output}")
print()
print(f"Output[0,0] = {output}")

**Verify.** The markdown walkthrough computed exactly this:
- patch · filter, element-wise sum = -1
- + bias 1 = 0
- ReLU(0) = 0

Got 0. Math checks.

## 4. Slide One Step Right

Now the top-left of the patch is at column 1 instead of column 0. Same filter, different patch.

In [ ]:
# Slide right by one column
patch2 = image[0:3, 1:4]
print("Patch (slid right by 1):")
print(patch2)
print()

elementwise = patch2 * filter_F
total = elementwise.sum()
z = total + bias
output = max(0, z)

print(f"Sum: {total}, + bias: {z}, ReLU: {output}")
print(f"Output[0,1] = {output}")

## 5. Build the Full Feature Map — Slide Everywhere

For a 5×5 input with a 3×3 filter, stride 1, no padding:

```
O = (5 - 3) / 1 + 1 = 3
```

So one filter produces a **3×3 feature map**. Loop through all 9 positions.

In [ ]:
def manual_convolve(image, filter_F, bias):
    """Apply a 3x3 filter to an image with stride 1, no padding. Pure NumPy."""
    H, W = image.shape
    F_size = filter_F.shape[0]
    out_size = (H - F_size) + 1   # stride 1, no padding

    output = np.zeros((out_size, out_size), dtype=np.float32)
    for i in range(out_size):
        for j in range(out_size):
            patch = image[i:i+F_size, j:j+F_size]
            output[i, j] = (patch * filter_F).sum() + bias
    return output

# Pre-activation feature map (before ReLU)
feature_map = manual_convolve(image, filter_F, bias)
print("Pre-activation feature map (z values):")
print(feature_map)

## 6. Verify Against PyTorch's `F.conv2d`

If our manual computation is correct, PyTorch's optimized convolution should produce the same numbers.

In [ ]:
# Convert to PyTorch tensors
# PyTorch expects shape (batch, channels, height, width)
image_t  = torch.tensor(image,  dtype=torch.float32).unsqueeze(0).unsqueeze(0)
filter_t = torch.tensor(filter_F, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
bias_t   = torch.tensor([bias], dtype=torch.float32)

# Run PyTorch convolution
torch_output = F.conv2d(image_t, filter_t, bias=bias_t, stride=1, padding=0)
torch_output_np = torch_output.squeeze().numpy()

print("PyTorch F.conv2d output:")
print(torch_output_np)
print()

print("Difference vs manual:", np.max(np.abs(feature_map - torch_output_np)))
print("All match within float precision:", np.allclose(feature_map, torch_output_np))

## 7. Apply ReLU — Element-Wise

ReLU(x) = max(0, x). It is applied independently to every cell in the feature map. Negative values become 0; positive values pass through unchanged.

In [ ]:
def relu(x):
    return np.maximum(0, x)

feature_map_relu = relu(feature_map)
print("Feature map after ReLU:")
print(feature_map_relu)
print()

# Verify against PyTorch
torch_relu = F.relu(torch_output).squeeze().numpy()
print("PyTorch F.relu output:")
print(torch_relu)
print()
print("Match:", np.allclose(feature_map_relu, torch_relu))

## 8. Manual 2×2 Max Pooling

Take a 2×2 region. Output the maximum value. Slide. Repeat.

For a 3×3 input with 2×2 pool size, stride 1: output is 2×2. (For stride 2, output would be smaller — but with a 3×3 input we cannot stride 2 cleanly.)

We will use **stride 1** here so we can see all four pooling windows.

In [ ]:
def max_pool_2x2(feature_map, stride=1):
    """Manual 2x2 max pooling. Pure NumPy."""
    H, W = feature_map.shape
    out_H = (H - 2) // stride + 1
    out_W = (W - 2) // stride + 1
    output = np.zeros((out_H, out_W), dtype=np.float32)
    for i in range(out_H):
        for j in range(out_W):
            region = feature_map[i*stride:i*stride+2, j*stride:j*stride+2]
            output[i, j] = region.max()
    return output

pooled = max_pool_2x2(feature_map_relu, stride=1)
print("After 2x2 max pooling (stride 1):")
print(pooled)
print()

# Verify against PyTorch
relu_t = torch.tensor(feature_map_relu, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
torch_pooled = F.max_pool2d(relu_t, kernel_size=2, stride=1).squeeze().numpy()
print("PyTorch F.max_pool2d output:")
print(torch_pooled)
print()
print("Match:", np.allclose(pooled, torch_pooled))

## 9. The Standard Choice — 2×2 Pool with Stride 2

In real CNNs, pooling almost always uses stride 2 to halve the spatial dimensions. With our 3×3 input that does not fit cleanly, so let us pool a bigger feature map.

Build a 4×4 feature map and pool it to 2×2.

In [ ]:
# A bigger toy feature map
big_fmap = np.array([
    [1, 2, 3, 0],
    [5, 6, 1, 2],
    [0, 3, 4, 4],
    [7, 1, 2, 9],
], dtype=np.float32)

print("Big feature map (4x4):")
print(big_fmap)
print()

# Manual stride-2 max pool: walks in 2-step jumps
def max_pool_2x2_stride2(feature_map):
    H, W = feature_map.shape
    out_H, out_W = H // 2, W // 2
    output = np.zeros((out_H, out_W), dtype=np.float32)
    for i in range(out_H):
        for j in range(out_W):
            region = feature_map[i*2:i*2+2, j*2:j*2+2]
            output[i, j] = region.max()
    return output

pooled_s2 = max_pool_2x2_stride2(big_fmap)
print("After 2x2 max pool, stride 2:")
print(pooled_s2)
print()
print("Spatial dimensions halved: 4x4 -> 2x2. Pooling has zero learnable parameters.")

## 10. Multiple Filters — How Depth Works

One filter detects one pattern. Real CNNs use **many filters** in parallel. Each filter produces its own feature map. The output has **depth** = number of filters.

Apply 4 filters to the 5×5 image — output shape: 3×3×4 (3×3 spatial, 4 channels).

In [ ]:
# Four filters — vertical edges, horizontal edges, "blob" detector, "anti-blob"
filter_v = np.array([[ 1, 0,-1], [ 1, 0,-1], [ 1, 0,-1]], dtype=np.float32)  # vertical edge
filter_h = np.array([[ 1, 1, 1], [ 0, 0, 0], [-1,-1,-1]], dtype=np.float32)  # horizontal edge
filter_b = np.array([[-1,-1,-1], [-1, 8,-1], [-1,-1,-1]], dtype=np.float32)  # center blob
filter_a = np.array([[ 1, 1, 1], [ 1,-8, 1], [ 1, 1, 1]], dtype=np.float32)  # anti-blob

filters = np.stack([filter_v, filter_h, filter_b, filter_a])
biases  = np.array([1.0, 0.5, 0.0, 0.0], dtype=np.float32)

print(f"Filters shape: {filters.shape}  (4 filters, each 3x3)")
print(f"Output spatial size will be: 3x3")
print(f"Output depth: 4 (one per filter)")
print()

# Apply each filter independently
output_3d = np.zeros((4, 3, 3), dtype=np.float32)
for k in range(4):
    output_3d[k] = relu(manual_convolve(image, filters[k], biases[k]))

print("Output volume (4 channels, each 3x3):\n")
for k, name in enumerate(['vertical-edge', 'horizontal-edge', 'center-blob', 'anti-blob']):
    print(f"Channel {k} ({name}):")
    print(output_3d[k])
    print()

**Key takeaway.** Output depth equals number of filters. Each filter learns to detect something different. The next conv layer's filters will *combine* these channels into more complex patterns — that's where the hierarchical learning comes from.

## 11. Parameter Counting — Verify the Formula

The formula from the markdown:

```
parameters_per_filter = F × F × D_in + 1   (the +1 is the bias)
total_parameters       = (F × F × D_in + 1) × K
```

For our 4-filter layer with 3×3 filters on a single-channel input:
```
per filter = 3 × 3 × 1 + 1 = 10
total      = 10 × 4 = 40
```

Verify by building this layer in PyTorch and counting parameters.

In [ ]:
import torch.nn as nn

# Build the same layer in PyTorch
conv_layer = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, bias=True)

# Count parameters
total_params = sum(p.numel() for p in conv_layer.parameters())
print(f"PyTorch reports: {total_params} parameters")
print(f"Manual formula:  (3 * 3 * 1 + 1) * 4 = {(3*3*1 + 1) * 4}")
print(f"Match: {total_params == (3*3*1 + 1) * 4}")
print()

# Show the breakdown
for name, p in conv_layer.named_parameters():
    print(f"  {name}: shape={tuple(p.shape)}, count={p.numel()}")
print()
print("Conv layer weights are 4 × (1 × 3 × 3) = 36 weights + 4 biases = 40 total.")

## 12. Two-Layer Comparison — Why Weight Sharing Wins

Compare the parameter count of:
- A **conv layer** with 16 filters of 3×3 on a depth-8 input
- An equivalent **fully-connected layer** with the same number of output channels

In [ ]:
# Conv layer: 16 filters of 3x3 on depth-8 input
conv_params = (3 * 3 * 8 + 1) * 16
print(f"Conv layer (3x3, in=8, out=16): {conv_params} parameters")

# FC layer: input 28x28x8 = 6272 inputs, 16 outputs
fc_input_size = 28 * 28 * 8
fc_output_size = 16
fc_params = fc_input_size * fc_output_size + fc_output_size
print(f"FC layer (28x28x8 -> 16):       {fc_params} parameters")
print()

ratio = fc_params / conv_params
print(f"FC has {ratio:.0f}x more parameters than the equivalent Conv.")
print()
print("This is why CNNs are tractable on images and MLPs are not.")

## 13. 1×1 Convolution — The Channel Mixer (Bonus)

A 1×1 filter sees one pixel — but it sees ALL the channels at that pixel. So it does not detect spatial patterns. It mixes channels.

Apply 2 filters of size 1×1 to a 4-channel input. Output has 2 channels.

In [ ]:
# Take the 4-channel output from earlier (4 channels, 3x3 spatial)
print(f"Input to 1x1 conv: {output_3d.shape}  (4 channels, 3x3 spatial)\n")

# 1x1 conv: each output channel is a learned weighted sum of the 4 input channels
# Two filters, each is a 4-vector (one weight per input channel)
filters_1x1 = np.array([
    [0.5, 0.5, 0.0, 0.0],   # filter 1: average of vertical + horizontal edges
    [0.0, 0.0, 1.0, -1.0],  # filter 2: difference of blob detectors
], dtype=np.float32)

# Apply: at each spatial position, do a dot product between input channels and the filter
def manual_1x1_conv(input_3d, filters_1x1):
    K = filters_1x1.shape[0]      # number of output filters
    H, W = input_3d.shape[1], input_3d.shape[2]
    output = np.zeros((K, H, W), dtype=np.float32)
    for k in range(K):
        for i in range(H):
            for j in range(W):
                output[k, i, j] = (input_3d[:, i, j] * filters_1x1[k]).sum()
    return output

output_1x1 = manual_1x1_conv(output_3d, filters_1x1)
print(f"After 1x1 conv: {output_1x1.shape}  (2 channels, 3x3 spatial)\n")

print("Channel 0 (avg of edge filters):")
print(output_1x1[0])
print()
print("Channel 1 (blob - anti-blob):")
print(output_1x1[1])

**Notice.** The spatial size (3×3) did not change. Only the depth changed: 4 → 2. That is the unique role of 1×1 convolutions: cheap channel mixing without touching spatial structure.

This is heavily used in modern architectures (ResNet bottlenecks, Inception modules, MobileNet's depthwise-separable convolutions).

## 14. Now Let's Train — Forward + Backward + Update

Above we built CNN's **forward pass** by hand. The mechanics of how an image becomes a feature map. Let's complete the picture with **training** — what happens when you call `loss.backward()` and `optimizer.step()`.

The unique CNN-specific feature in backprop is **weight-sharing gradient accumulation** — each filter weight collects gradient contributions from every position it was applied to. We will compute every gradient by hand, then verify against PyTorch autograd.

This mirrors `playbooks/ai/computer-vision/02_Concepts.md → Training a CNN — Numbers by Hand`.

### Setup

The smallest network that exercises every CNN-specific gradient.

```
Input X (3×3):           Filter W (2×2):
 1  0  1                  0.5  -0.3
 0  1  0                  0.1   0.4
 1  0  1

Bias b = 0.1
Target y = 1.5
Learning rate α = 0.1
```

The pipeline: `X → Conv → ReLU → sum → y_pred → loss`.

The `sum` is a stand-in for a global pooling head — gives one output for a scalar loss.

In [ ]:
# Setup
import numpy as np
np.set_printoptions(precision=4, suppress=True)

X_train = np.array([[1, 0, 1],
                    [0, 1, 0],
                    [1, 0, 1]], dtype=np.float32)

W = np.array([[ 0.5, -0.3],
              [ 0.1,  0.4]], dtype=np.float32)
b = 0.1
target = 1.5
alpha = 0.1

print(f"Input X:\n{X_train}")
print(f"Filter W:\n{W}")
print(f"Bias: {b}")
print(f"Target: {target}")
print(f"Learning rate: {alpha}")

### Forward Pass

Output spatial size: `(3 − 2)/1 + 1 = 2`. Feature map is 2×2.

In [ ]:
def conv_forward(X, W, b):
    """2D convolution, stride 1, no padding. Returns z (pre-activation)."""
    H, W_in = X.shape
    F = W.shape[0]
    out = H - F + 1
    z = np.zeros((out, out), dtype=np.float32)
    for i in range(out):
        for j in range(out):
            patch = X[i:i+F, j:j+F]
            z[i, j] = (patch * W).sum() + b
    return z

z = conv_forward(X_train, W, b)
h = np.maximum(0, z)              # ReLU
y_pred = h.sum()                  # sum reduction

print(f"z (pre-activation):\n{z}")
print(f"h (after ReLU):\n{h}")
print(f"y_pred = sum(h) = {y_pred}")

**Notice.** Two cells were killed by ReLU (the off-diagonal cells where z = -0.1 < 0). Those will receive **zero gradient** in the backward pass.

### Loss

In [ ]:
L = 0.5 * (y_pred - target)**2
print(f"L = 0.5 · (y_pred - target)² = 0.5 · ({y_pred} - {target})² = {L}")

### Backward Pass — The Weight-Sharing Trick

Every filter weight was applied at multiple positions during the forward pass. Its gradient is the **sum** of contributions from every position. Other positions where the filter sat that produced dead-ReLU cells contribute zero.

In [ ]:
# Step 1: gradient seed
dL_dy_pred = y_pred - target
print(f"∂L/∂y_pred = {dL_dy_pred}")

# Step 2: through the sum (every cell contributes equally)
dL_dh = np.full_like(h, dL_dy_pred)
print(f"∂L/∂h:\n{dL_dh}")

# Step 3: through ReLU (dead cells block gradient)
dL_dz = dL_dh * (z > 0).astype(np.float32)
print(f"∂L/∂z (after ReLU mask):\n{dL_dz}")

In [ ]:
# Step 4: bias gradient (sum of dL/dz, every cell contributes one +b)
dL_db = dL_dz.sum()
print(f"∂L/∂b = sum(∂L/∂z) = {dL_db}")

# Step 5: filter weight gradients — THE WEIGHT-SHARING ACCUMULATION
# z[i,j] = sum_uv W[u,v] * X[i+u, j+v] + b
# So  ∂L/∂W[u,v] = sum over (i,j) of ∂L/∂z[i,j] * X[i+u, j+v]
F = W.shape[0]
out = z.shape[0]
dL_dW = np.zeros_like(W)
for u in range(F):
    for v in range(F):
        for i in range(out):
            for j in range(out):
                dL_dW[u, v] += dL_dz[i, j] * X_train[i+u, j+v]

print(f"∂L/∂W (weight-shared accumulation):\n{dL_dW}")

**This is what makes CNN backprop different from MLP.** Each filter weight gets `out × out` gradient contributions — one per position where it was applied. PyTorch's `nn.Conv2d` does this for you when you call `loss.backward()`.

### Verify Against PyTorch Autograd

In [ ]:
import torch
import torch.nn.functional as F

X_t = torch.tensor(X_train, requires_grad=False).unsqueeze(0).unsqueeze(0)
W_t = torch.tensor(W, requires_grad=True).unsqueeze(0).unsqueeze(0)
b_t = torch.tensor([b], requires_grad=True)
target_t = torch.tensor(target)

# Forward in PyTorch
z_t = F.conv2d(X_t, W_t, bias=b_t, stride=1)
h_t = F.relu(z_t)
y_pred_t = h_t.sum()
L_t = 0.5 * (y_pred_t - target_t)**2

# Backward
L_t.backward()

print(f"PyTorch ∂L/∂W:\n{W_t.grad.squeeze().detach().numpy()}")
print(f"Manual  ∂L/∂W:\n{dL_dW}")
print(f"Match ∂L/∂W: {np.allclose(W_t.grad.squeeze().detach().numpy(), dL_dW)}")
print()
print(f"PyTorch ∂L/∂b: {b_t.grad.item()}")
print(f"Manual  ∂L/∂b: {dL_db}")
print(f"Match ∂L/∂b: {np.isclose(b_t.grad.item(), dL_db)}")

**Identical numbers.** PyTorch's autograd is doing the same chain rule + weight-sharing summation we just computed by hand.

### Update — One Step of Gradient Descent

In [ ]:
W_new = W - alpha * dL_dW
b_new = b - alpha * dL_db

print(f"W_new:\n{W_new}")
print(f"b_new: {b_new}")

### Train for 50 Cycles — Watch the Loss Drop

A real training loop runs this loop hundreds or thousands of times. Let's run 50 cycles and watch the loss converge.

In [ ]:
# Reset to initial weights
W_train = W.copy()
b_train = b

losses = []
EPOCHS = 50

for epoch in range(EPOCHS):
    # Forward
    z = conv_forward(X_train, W_train, b_train)
    h = np.maximum(0, z)
    y_pred = h.sum()

    # Loss
    L = 0.5 * (y_pred - target)**2
    losses.append(L)

    # Backward
    dL_dy = y_pred - target
    dL_dh = np.full_like(h, dL_dy)
    dL_dz = dL_dh * (z > 0).astype(np.float32)
    dL_db = dL_dz.sum()
    dL_dW = np.zeros_like(W_train)
    for u in range(F):
        for v in range(F):
            for i in range(out):
                for j in range(out):
                    dL_dW[u, v] += dL_dz[i, j] * X_train[i+u, j+v]

    # Update
    W_train -= alpha * dL_dW
    b_train -= alpha * dL_db

print(f"Loss at epoch 1:  {losses[0]:.4f}")
print(f"Loss at epoch 10: {losses[9]:.4f}")
print(f"Loss at epoch 25: {losses[24]:.4f}")
print(f"Loss at epoch 50: {losses[-1]:.6f}")
print()
print(f"Final y_pred: {y_pred:.4f}  (target = {target})")
print(f"Final W:\n{W_train}")
print(f"Final b: {b_train:.4f}")

In [ ]:
# Plot the loss curve
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN training loss — single example, weight-sharing gradient updates')
plt.grid(True, alpha=0.3)
plt.show()

print(f'Loss reduction: {losses[0]:.4f} → {losses[-1]:.6f}')

### What You Just Did

| Step | What you proved |
|---|---|
| Forward pass | Same as Section 5 — convolution by hand |
| Loss | Standard MSE; nothing CNN-specific |
| `∂L/∂y_pred → ∂L/∂h → ∂L/∂z` | Standard backprop through sum + ReLU; same as MLP |
| **`∂L/∂W` weight-sharing** | The unique CNN gradient — each filter weight is a SUM of contributions from every position |
| `∂L/∂b` | Sum of `∂L/∂z` since every position adds bias once |
| PyTorch verification | Manual gradients match `loss.backward()` exactly |
| Training loop | Loss drops, model learns to predict the target |

When you write `nn.Conv2d` + `loss.backward()` + `optimizer.step()` in PyTorch, you now know exactly what is happening underneath. The weight-sharing summation is the secret of CNN's parameter efficiency — and now it is no longer secret.

## What you just did

| Step | What you proved |
|---|---|
| Single slide | A convolution is dot product + bias + activation, exactly like every other neuron |
| Full feature map | The same filter applied at every position gives one feature map. Weight sharing in action. |
| PyTorch verification | `F.conv2d` produces the exact same numbers as the manual loop |
| ReLU | An element-wise non-linearity. Cheap and effective. |
| Max pooling | Reduces spatial size, keeps the strongest signal, has zero learnable parameters |
| Multi-filter conv | Output depth = number of filters. Each filter learns a different pattern. |
| Parameter count | Verified `(F × F × D_in + 1) × K` against PyTorch's actual count |
| Conv vs FC | A conv layer uses ~6,200x fewer parameters than the equivalent FC layer |
| 1×1 conv | Mixes channels without changing spatial size — the bottleneck trick |

When you write `nn.Conv2d(1, 16, kernel_size=3)` in PyTorch, you now know exactly what that layer does. No magic.

---

## Run the Full Pipeline

This notebook covered the **forward pass** of a CNN by hand. To see a complete CNN trained end-to-end on real digit images (MNIST), with backprop, full training loop, loss curves, and evaluation:

**[Deep Learning CNN on Colab](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Deep_Learning_CNN.ipynb)** — 34 cells covering convolution, pooling, full pipeline, MNIST training, and accuracy evaluation.

For the **deep learning foundations** that make CNN training work — perceptron, backprop, the chain rule:

**[Deep Learning From Scratch on Colab](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Deep_Learning_From_Scratch.ipynb)** — pure NumPy MLP trained by hand, then verified against PyTorch autograd.

---

**Next chapter:** [04 — How It Works](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/computer-vision/04_How_It_Works.md) — Receptive field, training diagnostics, transfer learning, augmentation.